In [3]:
import pandas as pd

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn)
df_nor = pd.read_parquet(path_northumbria)

for name, df in [("SEVERN", df_sev), ("NORTHUMBRIA", df_nor)]:
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}):\n  {list(df.columns)}")
    print(f"\nDate range: {df['valid_time'].min()} to {df['valid_time'].max()}")
    print(f"Number of unique dates: {df['valid_time'].nunique()}")
    print(f"Unique regions: {df['region'].unique()}")
    print(f"\nNull counts:\n{df.isnull().sum()}")
    print(f"\nFirst 3 rows (tp and sro columns only):")
    print(df[["valid_time", "tp_mean", "tp_max", "tp_sum",
              "sro_mean", "sro_max", "sro_sum",
              "swvl1_mean_mean", "swvl1_mean_max"]].head(3))
    print(f"\nBasic stats for key variables:")
    print(df[["tp_mean", "tp_max", "tp_sum",
              "sro_mean", "sro_max", "sro_sum",
              "swvl1_mean_mean", "swvl1_mean_max"]].describe())


  SEVERN
Shape: (3653, 44)
Columns (44):
  ['region', 'valid_time', 'u10_mean_mean', 'u10_mean_max', 'u10_mean_sum', 'v10_mean_mean', 'v10_mean_max', 'v10_mean_sum', 'd2m_mean_mean', 'd2m_mean_max', 'd2m_mean_sum', 't2m_mean_mean', 't2m_mean_max', 't2m_mean_sum', 'sp_mean_mean', 'sp_mean_max', 'sp_mean_sum', 'swvl1_mean_mean', 'swvl1_mean_max', 'swvl1_mean_sum', 'u10_max_mean', 'u10_max_max', 'u10_max_sum', 'v10_max_mean', 'v10_max_max', 'v10_max_sum', 'd2m_max_mean', 'd2m_max_max', 'd2m_max_sum', 't2m_max_mean', 't2m_max_max', 't2m_max_sum', 'sp_max_mean', 'sp_max_max', 'sp_max_sum', 'swvl1_max_mean', 'swvl1_max_max', 'swvl1_max_sum', 'sro_mean', 'sro_max', 'sro_sum', 'tp_mean', 'tp_max', 'tp_sum']

Date range: 2015-01-01 00:00:00 to 2024-12-31 00:00:00
Number of unique dates: 3653
Unique regions: ['severn']

Null counts:
region             0
valid_time         0
u10_mean_mean      0
u10_mean_max       0
u10_mean_sum       0
v10_mean_mean      0
v10_mean_max       0
v10_mean_sum     

In [4]:
# ── Step 2: Rolling window precipitation accumulations ──
# For each region, compute rolling sums of tp_mean over 5, 10, 15-day windows,
# then take the MAX of each rolling sum across the full 10-year time series.
# This captures the worst sustained rainfall event the region experienced.

import pandas as pd

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn).sort_values("valid_time").reset_index(drop=True)
df_nor = pd.read_parquet(path_northumbria).sort_values("valid_time").reset_index(drop=True)

results = {}

for name, df in [("severn", df_sev), ("northumbria", df_nor)]:
    feats = {}

    for window in [5, 10, 15]:
        # Rolling sum of the region-average daily precipitation
        rolling_sum = df["tp_mean"].rolling(window=window).sum()
        feats[f"max_rolling_{window}d_tp_mean"] = rolling_sum.max()

        # Also do it for tp_max (the daily spatial max across the region)
        rolling_sum_max = df["tp_max"].rolling(window=window).sum()
        feats[f"max_rolling_{window}d_tp_max"] = rolling_sum_max.max()

    results[name] = feats

# Display results
df_tp_feats = pd.DataFrame(results).T
df_tp_feats.index.name = "region"
print("Rolling precipitation features (max of rolling sums):\n")
print(df_tp_feats.to_string())

Rolling precipitation features (max of rolling sums):

             max_rolling_5d_tp_mean  max_rolling_5d_tp_max  max_rolling_10d_tp_mean  max_rolling_10d_tp_max  max_rolling_15d_tp_mean  max_rolling_15d_tp_max
region                                                                                                                                                      
severn                     0.078810               0.120211                 0.103284                0.181541                 0.111550                0.203687
northumbria                0.067703               0.118542                 0.092166                0.160062                 0.123695                0.220147


In [5]:
# ── Step 3: Rolling window surface runoff accumulations ──

import pandas as pd

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn).sort_values("valid_time").reset_index(drop=True)
df_nor = pd.read_parquet(path_northumbria).sort_values("valid_time").reset_index(drop=True)

results = {}

for name, df in [("severn", df_sev), ("northumbria", df_nor)]:
    feats = {}

    for window in [5, 10, 15]:
        # Rolling sum of region-average daily surface runoff
        rolling_sum = df["sro_mean"].rolling(window=window).sum()
        feats[f"max_rolling_{window}d_sro_mean"] = rolling_sum.max()

        # Rolling sum of daily spatial max surface runoff
        rolling_sum_max = df["sro_max"].rolling(window=window).sum()
        feats[f"max_rolling_{window}d_sro_max"] = rolling_sum_max.max()

    results[name] = feats

df_sro_feats = pd.DataFrame(results).T
df_sro_feats.index.name = "region"
print("Rolling surface runoff features (max of rolling sums):\n")
print(df_sro_feats.to_string())

Rolling surface runoff features (max of rolling sums):

             max_rolling_5d_sro_mean  max_rolling_5d_sro_max  max_rolling_10d_sro_mean  max_rolling_10d_sro_max  max_rolling_15d_sro_mean  max_rolling_15d_sro_max
region                                                                                                                                                            
severn                      0.004860                0.016703                  0.006824                 0.023256                  0.007093                 0.024963
northumbria                 0.010376                0.051975                  0.012138                 0.062550                  0.015226                 0.083410


In [6]:
# ── Step 4: Soil moisture saturation and depletion extremes ──

import pandas as pd

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn).sort_values("valid_time").reset_index(drop=True)
df_nor = pd.read_parquet(path_northumbria).sort_values("valid_time").reset_index(drop=True)

results = {}

for name, df in [("severn", df_sev), ("northumbria", df_nor)]:
    feats = {}

    # 7-day rolling average of region-mean soil moisture
    roll_7d_mean = df["swvl1_mean_mean"].rolling(window=7).mean()
    feats["max_rolling_7d_swvl1_mean"] = roll_7d_mean.max()   # peak saturation
    feats["min_rolling_7d_swvl1_mean"] = roll_7d_mean.min()   # peak depletion

    # 7-day rolling average of region-max soil moisture
    roll_7d_max = df["swvl1_mean_max"].rolling(window=7).mean()
    feats["max_rolling_7d_swvl1_spatial_max"] = roll_7d_max.max()
    feats["min_rolling_7d_swvl1_spatial_max"] = roll_7d_max.min()

    # Also capture the overall mean and std for context
    feats["swvl1_mean_overall"] = df["swvl1_mean_mean"].mean()
    feats["swvl1_std_overall"] = df["swvl1_mean_mean"].std()

    results[name] = feats

df_swvl_feats = pd.DataFrame(results).T
df_swvl_feats.index.name = "region"
print("Soil moisture features:\n")
print(df_swvl_feats.to_string())

Soil moisture features:

             max_rolling_7d_swvl1_mean  min_rolling_7d_swvl1_mean  max_rolling_7d_swvl1_spatial_max  min_rolling_7d_swvl1_spatial_max  swvl1_mean_overall  swvl1_std_overall
region                                                                                                                                                                      
severn                        0.415873                   0.163217                          0.502811                          0.272563            0.350764           0.053853
northumbria                   0.407701                   0.160587                          0.504674                          0.263348            0.349145           0.042669


In [7]:
# ── Step 5: Percentile-based features for tp and sro ──

import pandas as pd
import numpy as np

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn)
df_nor = pd.read_parquet(path_northumbria)

results = {}

for name, df in [("severn", df_sev), ("northumbria", df_nor)]:
    feats = {}

    for var in ["tp_mean", "tp_max", "sro_mean", "sro_max"]:
        feats[f"{var}_p50"] = np.percentile(df[var], 50)
        feats[f"{var}_p90"] = np.percentile(df[var], 90)
        feats[f"{var}_p95"] = np.percentile(df[var], 95)
        feats[f"{var}_p99"] = np.percentile(df[var], 99)
        feats[f"{var}_mean"] = df[var].mean()
        feats[f"{var}_std"] = df[var].std()

    results[name] = feats

df_pctl_feats = pd.DataFrame(results).T
df_pctl_feats.index.name = "region"

print("Percentile features:\n")
# Print in a readable format, grouped by variable
for var in ["tp_mean", "tp_max", "sro_mean", "sro_max"]:
    cols = [c for c in df_pctl_feats.columns if c.startswith(var)]
    print(f"\n--- {var} ---")
    print(df_pctl_feats[cols].to_string())

Percentile features:


--- tp_mean ---
             tp_mean_p50  tp_mean_p90  tp_mean_p95  tp_mean_p99  tp_mean_mean  tp_mean_std
region                                                                                    
severn          0.000713     0.007016     0.009671     0.016625      0.002321     0.003606
northumbria     0.001099     0.006886     0.010111     0.018537      0.002585     0.003839

--- tp_max ---
             tp_max_p50  tp_max_p90  tp_max_p95  tp_max_p99  tp_max_mean  tp_max_std
region                                                                              
severn         0.002365    0.012785    0.016768    0.027476     0.004792    0.006091
northumbria    0.003063    0.012718    0.017217    0.028316     0.005156    0.006197

--- sro_mean ---
             sro_mean_p50  sro_mean_p90  sro_mean_p95  sro_mean_p99  sro_mean_mean  sro_mean_std
region                                                                                          
severn           0.000006    

In [8]:
# ── Step 6: Seasonal extreme features ──

import pandas as pd
import numpy as np

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"

df_sev = pd.read_parquet(path_severn)
df_nor = pd.read_parquet(path_northumbria)

# Map each month to a meteorological season
def get_season(month):
    if month in [12, 1, 2]:
        return "DJF"  # winter
    elif month in [3, 4, 5]:
        return "MAM"  # spring
    elif month in [6, 7, 8]:
        return "JJA"  # summer
    else:
        return "SON"  # autumn

results = {}

for name, df in [("severn", df_sev), ("northumbria", df_nor)]:
    df = df.copy()
    df["season"] = df["valid_time"].dt.month.map(get_season)

    feats = {}

    for var in ["tp_mean", "tp_max", "sro_mean", "sro_max", "swvl1_mean_mean"]:
        seasonal = df.groupby("season")[var].agg(["mean", "max", "std"])
        for season in ["DJF", "MAM", "JJA", "SON"]:
            feats[f"{var}_{season}_mean"] = seasonal.loc[season, "mean"]
            feats[f"{var}_{season}_max"] = seasonal.loc[season, "max"]
            feats[f"{var}_{season}_std"] = seasonal.loc[season, "std"]

    results[name] = feats

df_season_feats = pd.DataFrame(results).T
df_season_feats.index.name = "region"

print(f"Seasonal features shape: {df_season_feats.shape}")
print(f"Columns: {list(df_season_feats.columns)}\n")

# Print a subset for readability: tp_mean by season
print("--- tp_mean seasonal breakdown ---")
tp_cols = [c for c in df_season_feats.columns if c.startswith("tp_mean_")]
print(df_season_feats[tp_cols].T.to_string())

print("\n--- sro_mean seasonal breakdown ---")
sro_cols = [c for c in df_season_feats.columns if c.startswith("sro_mean_")]
print(df_season_feats[sro_cols].T.to_string())

print("\n--- swvl1_mean_mean seasonal breakdown ---")
swvl_cols = [c for c in df_season_feats.columns if c.startswith("swvl1_mean_mean_")]
print(df_season_feats[swvl_cols].T.to_string())

Seasonal features shape: (2, 60)
Columns: ['tp_mean_DJF_mean', 'tp_mean_DJF_max', 'tp_mean_DJF_std', 'tp_mean_MAM_mean', 'tp_mean_MAM_max', 'tp_mean_MAM_std', 'tp_mean_JJA_mean', 'tp_mean_JJA_max', 'tp_mean_JJA_std', 'tp_mean_SON_mean', 'tp_mean_SON_max', 'tp_mean_SON_std', 'tp_max_DJF_mean', 'tp_max_DJF_max', 'tp_max_DJF_std', 'tp_max_MAM_mean', 'tp_max_MAM_max', 'tp_max_MAM_std', 'tp_max_JJA_mean', 'tp_max_JJA_max', 'tp_max_JJA_std', 'tp_max_SON_mean', 'tp_max_SON_max', 'tp_max_SON_std', 'sro_mean_DJF_mean', 'sro_mean_DJF_max', 'sro_mean_DJF_std', 'sro_mean_MAM_mean', 'sro_mean_MAM_max', 'sro_mean_MAM_std', 'sro_mean_JJA_mean', 'sro_mean_JJA_max', 'sro_mean_JJA_std', 'sro_mean_SON_mean', 'sro_mean_SON_max', 'sro_mean_SON_std', 'sro_max_DJF_mean', 'sro_max_DJF_max', 'sro_max_DJF_std', 'sro_max_MAM_mean', 'sro_max_MAM_max', 'sro_max_MAM_std', 'sro_max_JJA_mean', 'sro_max_JJA_max', 'sro_max_JJA_std', 'sro_max_SON_mean', 'sro_max_SON_max', 'sro_max_SON_std', 'swvl1_mean_mean_DJF_mean', '

In [9]:
# ── Step 7: Assemble all ERA5 features and save ──

import pandas as pd
import numpy as np

path_severn = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily.parquet"
path_northumbria = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_northumbria_daily.parquet"
out_dir = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

df_sev = pd.read_parquet(path_severn).sort_values("valid_time").reset_index(drop=True)
df_nor = pd.read_parquet(path_northumbria).sort_values("valid_time").reset_index(drop=True)

def get_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"

def build_era5_features(df, region_name):
    feats = {"region": region_name}

    # --- Rolling window features (tp and sro) ---
    for var in ["tp_mean", "tp_max", "sro_mean", "sro_max"]:
        for window in [5, 10, 15]:
            rolling_sum = df[var].rolling(window=window).sum()
            feats[f"max_rolling_{window}d_{var}"] = rolling_sum.max()

    # --- Soil moisture extremes ---
    roll_7d_mean = df["swvl1_mean_mean"].rolling(window=7).mean()
    feats["max_rolling_7d_swvl1_mean"] = roll_7d_mean.max()
    feats["min_rolling_7d_swvl1_mean"] = roll_7d_mean.min()

    roll_7d_max = df["swvl1_mean_max"].rolling(window=7).mean()
    feats["max_rolling_7d_swvl1_spatial_max"] = roll_7d_max.max()
    feats["min_rolling_7d_swvl1_spatial_max"] = roll_7d_max.min()

    feats["swvl1_mean_overall"] = df["swvl1_mean_mean"].mean()
    feats["swvl1_std_overall"] = df["swvl1_mean_mean"].std()

    # --- Percentile features ---
    for var in ["tp_mean", "tp_max", "sro_mean", "sro_max"]:
        for p in [50, 90, 95, 99]:
            feats[f"{var}_p{p}"] = np.percentile(df[var], p)
        feats[f"{var}_overall_mean"] = df[var].mean()
        feats[f"{var}_overall_std"] = df[var].std()

    # --- Seasonal extremes ---
    df_tmp = df.copy()
    df_tmp["season"] = df_tmp["valid_time"].dt.month.map(get_season)
    for var in ["tp_mean", "tp_max", "sro_mean", "sro_max", "swvl1_mean_mean"]:
        seasonal = df_tmp.groupby("season")[var].agg(["mean", "max", "std"])
        for season in ["DJF", "MAM", "JJA", "SON"]:
            feats[f"{var}_{season}_mean"] = seasonal.loc[season, "mean"]
            feats[f"{var}_{season}_max"] = seasonal.loc[season, "max"]
            feats[f"{var}_{season}_std"] = seasonal.loc[season, "std"]

    return feats

# Build feature rows
feats_sev = build_era5_features(df_sev, "severn")
feats_nor = build_era5_features(df_nor, "northumbria")

# Combine into a single DataFrame
df_era5_feats = pd.DataFrame([feats_sev, feats_nor])
df_era5_feats.to_parquet(f"{out_dir}\\era5_features.parquet", index=False)

# Verification
print(f"Shape: {df_era5_feats.shape}")
print(f"Columns ({len(df_era5_feats.columns)}):\n{list(df_era5_feats.columns)}")
print(f"\nNull count: {df_era5_feats.isnull().sum().sum()}")
print(f"\nFull table:\n{df_era5_feats.T.to_string()}")

Shape: (2, 103)
Columns (103):
['region', 'max_rolling_5d_tp_mean', 'max_rolling_10d_tp_mean', 'max_rolling_15d_tp_mean', 'max_rolling_5d_tp_max', 'max_rolling_10d_tp_max', 'max_rolling_15d_tp_max', 'max_rolling_5d_sro_mean', 'max_rolling_10d_sro_mean', 'max_rolling_15d_sro_mean', 'max_rolling_5d_sro_max', 'max_rolling_10d_sro_max', 'max_rolling_15d_sro_max', 'max_rolling_7d_swvl1_mean', 'min_rolling_7d_swvl1_mean', 'max_rolling_7d_swvl1_spatial_max', 'min_rolling_7d_swvl1_spatial_max', 'swvl1_mean_overall', 'swvl1_std_overall', 'tp_mean_p50', 'tp_mean_p90', 'tp_mean_p95', 'tp_mean_p99', 'tp_mean_overall_mean', 'tp_mean_overall_std', 'tp_max_p50', 'tp_max_p90', 'tp_max_p95', 'tp_max_p99', 'tp_max_overall_mean', 'tp_max_overall_std', 'sro_mean_p50', 'sro_mean_p90', 'sro_mean_p95', 'sro_mean_p99', 'sro_mean_overall_mean', 'sro_mean_overall_std', 'sro_max_p50', 'sro_max_p90', 'sro_max_p95', 'sro_max_p99', 'sro_max_overall_mean', 'sro_max_overall_std', 'tp_mean_DJF_mean', 'tp_mean_DJF_max'

In [10]:
# ── Step 8: Load and inspect cleaned terrain data ──

import xarray as xr

path_sev = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_severn_clean.nc"
path_nor = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_northumbria_clean.nc"

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Dimensions: {dict(ds.dims)}")
    print(f"\nVariables: {list(ds.data_vars)}")
    print(f"\nCoordinates: {list(ds.coords)}")

    # Check a few key variables
    for var in ds.data_vars:
        v = ds[var]
        print(f"\n  {var}: dtype={v.dtype}, shape={v.shape}")

    # Count valid pixels
    if "valid_pixel" in ds.data_vars:
        n_valid = ds["valid_pixel"].sum().compute().item()
        n_total = ds["valid_pixel"].size
        print(f"\nValid pixels: {n_valid:,} / {n_total:,} ({100*n_valid/n_total:.1f}%)")

    # Quick check on key variables for a small sample
    print("\n--- Sample stats (computed on a small chunk) ---")
    sample = ds.isel(x=slice(0, 500), y=slice(0, 500))
    for var in ["dtm", "dtm_zscore", "log_flow_acc", "rciw", "waw", "imd", "clc_type"]:
        if var in ds.data_vars:
            vals = sample[var].values.ravel()
            non_nan = vals[~np.isnan(vals)] if np.issubdtype(vals.dtype, np.floating) else vals
            print(f"  {var}: min={np.nanmin(vals):.2f}, max={np.nanmax(vals):.2f}, "
                  f"NaN%={100*np.isnan(vals).sum()/len(vals):.1f}%")

    ds.close()


  SEVERN
Dimensions: {'y': 10249, 'x': 8192}

Variables: ['dtm', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'log_flow_acc', 'dtm_zscore', 'valid_pixel']

Coordinates: ['y', 'x', 'spatial_ref']

  dtm: dtype=float32, shape=(10249, 8192)

  flow_acc: dtype=float64, shape=(10249, 8192)

  imd: dtype=float32, shape=(10249, 8192)

  waw: dtype=float32, shape=(10249, 8192)

  rciw: dtype=float32, shape=(10249, 8192)

  clc_type: dtype=float32, shape=(10249, 8192)

  risk_0_2m: dtype=int8, shape=(10249, 8192)

  risk_0_3m: dtype=int8, shape=(10249, 8192)

  risk_0_6m: dtype=int8, shape=(10249, 8192)

  risk_0_9m: dtype=int8, shape=(10249, 8192)

  risk_1_2m: dtype=int8, shape=(10249, 8192)

  log_flow_acc: dtype=float32, shape=(10249, 8192)

  dtm_zscore: dtype=float32, shape=(10249, 8192)

  valid_pixel: dtype=bool, shape=(10249, 8192)


C:\Users\jackp\AppData\Local\Temp\ipykernel_8280\3968142566.py:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Dimensions: {dict(ds.dims)}")



Valid pixels: 35,278,130 / 83,959,808 (42.0%)

--- Sample stats (computed on a small chunk) ---
  dtm: min=nan, max=nan, NaN%=100.0%
  dtm_zscore: min=nan, max=nan, NaN%=100.0%
  log_flow_acc: min=nan, max=nan, NaN%=100.0%
  rciw: min=nan, max=nan, NaN%=100.0%
  waw: min=nan, max=nan, NaN%=100.0%
  imd: min=nan, max=nan, NaN%=100.0%
  clc_type: min=nan, max=nan, NaN%=100.0%


C:\Users\jackp\AppData\Local\Temp\ipykernel_8280\3968142566.py:36: RuntimeWarning: All-NaN slice encountered
  print(f"  {var}: min={np.nanmin(vals):.2f}, max={np.nanmax(vals):.2f}, "



  NORTHUMBRIA
Dimensions: {'y': 8327, 'x': 5527}

Variables: ['dtm', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'log_flow_acc', 'dtm_zscore', 'valid_pixel']

Coordinates: ['y', 'x', 'spatial_ref']

  dtm: dtype=float32, shape=(8327, 5527)

  flow_acc: dtype=float64, shape=(8327, 5527)

  imd: dtype=float32, shape=(8327, 5527)

  waw: dtype=float32, shape=(8327, 5527)

  rciw: dtype=float32, shape=(8327, 5527)

  clc_type: dtype=float32, shape=(8327, 5527)

  risk_0_2m: dtype=int8, shape=(8327, 5527)

  risk_0_3m: dtype=int8, shape=(8327, 5527)

  risk_0_6m: dtype=int8, shape=(8327, 5527)

  risk_0_9m: dtype=int8, shape=(8327, 5527)

  risk_1_2m: dtype=int8, shape=(8327, 5527)

  log_flow_acc: dtype=float32, shape=(8327, 5527)

  dtm_zscore: dtype=float32, shape=(8327, 5527)

  valid_pixel: dtype=bool, shape=(8327, 5527)

Valid pixels: 21,391,824 / 46,023,329 (46.5%)

--- Sample stats (computed on a small chunk) ---
  

C:\Users\jackp\AppData\Local\Temp\ipykernel_8280\3968142566.py:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Dimensions: {dict(ds.dims)}")
C:\Users\jackp\AppData\Local\Temp\ipykernel_8280\3968142566.py:36: RuntimeWarning: All-NaN slice encountered
  print(f"  {var}: min={np.nanmin(vals):.2f}, max={np.nanmax(vals):.2f}, "


  waw: min=nan, max=nan, NaN%=100.0%
  imd: min=nan, max=nan, NaN%=100.0%
  clc_type: min=nan, max=nan, NaN%=100.0%


In [11]:
# ── Step 8b: Sample from valid pixels ──

import xarray as xr
import numpy as np

path_sev = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_severn_clean.nc"
path_nor = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_northumbria_clean.nc"

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})

    # Sample from the center of the grid
    cy, cx = ds.sizes["y"] // 2, ds.sizes["x"] // 2
    sample = ds.isel(y=slice(cy, cy + 500), x=slice(cx, cx + 500)).compute()

    print(f"\n{'='*60}")
    print(f"  {name} (center sample y={cy}:{cy+500}, x={cx}:{cx+500})")
    print(f"{'='*60}")

    for var in ["dtm", "dtm_zscore", "log_flow_acc", "rciw", "waw", "imd", "clc_type"]:
        if var in sample.data_vars:
            vals = sample[var].values.ravel()
            n_nan = np.isnan(vals).sum() if np.issubdtype(vals.dtype, np.floating) else 0
            non_nan = vals[~np.isnan(vals)] if np.issubdtype(vals.dtype, np.floating) else vals
            if len(non_nan) > 0:
                unique_count = len(np.unique(non_nan))
                print(f"  {var}: min={np.nanmin(non_nan):.2f}, max={np.nanmax(non_nan):.2f}, "
                      f"NaN%={100*n_nan/len(vals):.1f}%, unique_values={unique_count}")
            else:
                print(f"  {var}: ALL NaN in this sample")

    # Check risk distribution in sample
    print("\n  Risk distribution (risk_0_2m) in sample:")
    risk_vals = sample["risk_0_2m"].values.ravel()
    unique, counts = np.unique(risk_vals, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"    class {u}: {c:,} ({100*c/len(risk_vals):.1f}%)")

    ds.close()


  SEVERN (center sample y=5124:5624, x=4096:4596)
  dtm: min=99.00, max=1236.00, NaN%=0.0%, unique_values=1125
  dtm_zscore: min=-1.36, max=0.21, NaN%=0.0%, unique_values=1125
  log_flow_acc: min=3.04, max=18.64, NaN%=0.0%, unique_values=23551
  rciw: min=2.00, max=2.00, NaN%=99.7%, unique_values=1
  waw: min=0.00, max=4.00, NaN%=0.0%, unique_values=5
  imd: min=0.00, max=100.00, NaN%=0.0%, unique_values=100
  clc_type: min=112.00, max=322.00, NaN%=0.0%, unique_values=9

  Risk distribution (risk_0_2m) in sample:
    class 0: 212,931 (85.2%)
    class 1: 3,306 (1.3%)
    class 2: 7,528 (3.0%)
    class 3: 2,976 (1.2%)
    class 4: 23,259 (9.3%)

  NORTHUMBRIA (center sample y=4163:4663, x=2763:3263)
  dtm: min=313.00, max=1586.00, NaN%=0.0%, unique_values=1198
  dtm_zscore: min=-1.08, max=-0.25, NaN%=0.0%, unique_values=1198
  log_flow_acc: min=3.04, max=16.47, NaN%=0.0%, unique_values=123495
  rciw: ALL NaN in this sample
  waw: min=0.00, max=4.00, NaN%=0.0%, unique_values=3
  imd: m

In [12]:
# ── Step 9: Terrain feature engineering ──
# This step inspects the full ranges and applies fixes.
# We do NOT modify the NetCDF files (they are large).
# Instead we define the transformations and will apply them during
# the flattening step (Step 10).

import xarray as xr
import numpy as np

path_sev = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_severn_clean.nc"
path_nor = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_northumbria_clean.nc"

# ── Part A: Check full imd range to confirm the 0-255 issue ──
print("="*60)
print("  PART A: Full imd range check")
print("="*60)

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    imd_min = ds["imd"].min().compute().item()
    imd_max = ds["imd"].max().compute().item()
    print(f"  {name} imd range: {imd_min} to {imd_max}")
    ds.close()

# ── Part B: Check full rciw value counts ──
print(f"\n{'='*60}")
print("  PART B: rciw value distribution")
print("="*60)

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    valid_mask = ds["valid_pixel"].values
    rciw_vals = ds["rciw"].values
    
    # Count among valid pixels only
    valid_rciw = rciw_vals[valid_mask]
    n_valid = len(valid_rciw)
    n_nan = np.isnan(valid_rciw).sum()
    non_nan = valid_rciw[~np.isnan(valid_rciw)]
    
    print(f"\n  {name} (valid pixels only):")
    print(f"    Total valid pixels: {n_valid:,}")
    print(f"    rciw NaN: {n_nan:,} ({100*n_nan/n_valid:.2f}%)")
    if len(non_nan) > 0:
        unique, counts = np.unique(non_nan, return_counts=True)
        for u, c in zip(unique, counts):
            print(f"    rciw={int(u)}: {c:,} ({100*c/n_valid:.3f}%)")
    ds.close()

# ── Part C: Check full clc_type unique values ──
print(f"\n{'='*60}")
print("  PART C: clc_type unique values")
print("="*60)

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    valid_mask = ds["valid_pixel"].values
    clc_vals = ds["clc_type"].values[valid_mask]
    clc_valid = clc_vals[~np.isnan(clc_vals)]
    unique = np.unique(clc_valid)
    print(f"  {name}: {len(unique)} unique classes: {sorted(unique.astype(int).tolist())}")
    ds.close()

  PART A: Full imd range check
  SEVERN imd range: 0.0 to 100.0
  NORTHUMBRIA imd range: 0.0 to 255.0

  PART B: rciw value distribution

  SEVERN (valid pixels only):
    Total valid pixels: 35,278,130
    rciw NaN: 35,138,961 (99.61%)
    rciw=2: 95,767 (0.271%)
    rciw=3: 43,402 (0.123%)

  NORTHUMBRIA (valid pixels only):
    Total valid pixels: 21,391,824
    rciw NaN: 21,268,542 (99.42%)
    rciw=2: 97,325 (0.455%)
    rciw=3: 25,957 (0.121%)

  PART C: clc_type unique values
  SEVERN: 31 unique classes: [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142, 211, 222, 231, 242, 243, 311, 312, 313, 321, 322, 324, 333, 411, 412, 421, 423, 511, 512, 522, 523]
  NORTHUMBRIA: 30 unique classes: [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142, 211, 231, 243, 311, 312, 313, 321, 322, 324, 332, 333, 411, 412, 421, 423, 511, 512, 522, 523]


In [15]:
# ── Step 10 (chunked): Process terrain in strips, save incrementally ──

import xarray as xr
import numpy as np
import pandas as pd
import gc
import os

out_dir = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

terrain_paths = {
    "severn": r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_severn_clean.nc",
    "northumbria": r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\terrain_northumbria_clean.nc",
}

terrain_vars = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "rciw", "clc_type",
    "risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"
]

CHUNK_SIZE = 500  # rows of the y-axis per chunk

for region, path in terrain_paths.items():
    print(f"\n{'='*60}")
    print(f"  Processing {region.upper()}")
    print(f"{'='*60}")

    ds = xr.open_dataset(path)
    ny, nx = ds.sizes["y"], ds.sizes["x"]
    x_vals = ds.coords["x"].values
    print(f"  Grid: {ny} x {nx}, processing in chunks of {CHUNK_SIZE} rows")

    out_path = f"{out_dir}\\{region}_terrain_ready.parquet"
    all_chunks = []
    total_valid = 0

    for y_start in range(0, ny, CHUNK_SIZE):
        y_end = min(y_start + CHUNK_SIZE, ny)
        chunk = ds.isel(y=slice(y_start, y_end))

        # Load valid_pixel for this strip
        vp = chunk["valid_pixel"].values  # shape (chunk_h, nx)
        n_valid = vp.sum()
        if n_valid == 0:
            continue

        # Build coordinate arrays for valid pixels
        y_chunk = chunk.coords["y"].values
        y_grid, x_grid = np.meshgrid(y_chunk, x_vals, indexing="ij")

        data = {
            "proj_y": y_grid[vp].astype(np.float32),
            "proj_x": x_grid[vp].astype(np.float32),
        }

        for var in terrain_vars:
            data[var] = chunk[var].values[vp]

        df_chunk = pd.DataFrame(data)
        del data, y_grid, x_grid, vp
        gc.collect()

        # Apply fixes
        df_chunk["is_waterway"] = np.where(
            np.isnan(df_chunk["rciw"]), 0, 1
        ).astype(np.int8)
        df_chunk.drop(columns=["rciw"], inplace=True)

        df_chunk["imd"] = df_chunk["imd"].clip(0, 100).astype(np.float32)
        df_chunk["clc_type"] = df_chunk["clc_type"].fillna(0).astype(np.int16)
        df_chunk["waw"] = df_chunk["waw"].fillna(0).astype(np.int8)
        df_chunk["dtm_zscore"] = df_chunk["dtm_zscore"].astype(np.float32)
        df_chunk["log_flow_acc"] = df_chunk["log_flow_acc"].astype(np.float32)

        all_chunks.append(df_chunk)
        total_valid += len(df_chunk)

        # Print progress every 5000 rows
        if y_start % 5000 == 0:
            print(f"    y={y_start}-{y_end}: {n_valid:,} valid pixels "
                  f"(cumulative: {total_valid:,})")

        del df_chunk
        gc.collect()

    ds.close()

    # Concatenate all chunks
    print(f"  Concatenating {len(all_chunks)} chunks...")
    df = pd.concat(all_chunks, ignore_index=True)
    del all_chunks
    gc.collect()

    # Validation
    print(f"  Final shape: {df.shape}")
    null_counts = df.isnull().sum()
    if null_counts.sum() == 0:
        print("  No nulls!")
    else:
        print(f"  Nulls:\n{null_counts[null_counts > 0]}")

    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    print(f"  imd range: {df['imd'].min()} to {df['imd'].max()}")
    print(f"  is_waterway counts: 0={int((df['is_waterway']==0).sum()):,}, "
          f"1={int((df['is_waterway']==1).sum()):,}")
    print(f"  Columns: {list(df.columns)}")

    # Save
    df.to_parquet(out_path, index=False)
    print(f"  Saved to: {out_path}")

    del df
    gc.collect()


  Processing SEVERN
  Grid: 10249 x 8192, processing in chunks of 500 rows
    y=0-500: 349,071 valid pixels (cumulative: 349,071)
    y=5000-5500: 3,041,809 valid pixels (cumulative: 16,815,658)
    y=10000-10249: 319,629 valid pixels (cumulative: 35,278,130)
  Concatenating 21 chunks...
  Final shape: (35278130, 13)
  No nulls!
  Memory: 1.02 GB
  imd range: 0.0 to 100.0
  is_waterway counts: 0=35,138,961, 1=139,169
  Columns: ['proj_y', 'proj_x', 'dtm_zscore', 'log_flow_acc', 'imd', 'waw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'is_waterway']
  Saved to: C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\severn_terrain_ready.parquet

  Processing NORTHUMBRIA
  Grid: 8327 x 5527, processing in chunks of 500 rows
    y=0-500: 298,214 valid pixels (cumulative: 298,214)
    y=5000-5500: 1,786,280 valid pixels (cumulative: 18,163,115)
  Concatenating 17 chunks...
  Final shape: (21391824, 13)
  No nulls!
  Memory: 0.62 GB
  imd range: 0.0 to 100

In [16]:
# ── Step 11: Final validation ──

import pandas as pd
import numpy as np

out_dir = r"C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned"

# Load all three output files
df_sev = pd.read_parquet(f"{out_dir}\\severn_terrain_ready.parquet")
df_nor = pd.read_parquet(f"{out_dir}\\northumbria_terrain_ready.parquet")
df_era5 = pd.read_parquet(f"{out_dir}\\era5_features.parquet")

print("="*60)
print("  FILE SUMMARY")
print("="*60)
print(f"  severn_terrain_ready.parquet:      {df_sev.shape}")
print(f"  northumbria_terrain_ready.parquet:  {df_nor.shape}")
print(f"  era5_features.parquet:             {df_era5.shape}")

# ── Check 1: No nulls anywhere ──
print(f"\n{'='*60}")
print("  CHECK 1: Null counts")
print("="*60)
for name, df in [("Severn terrain", df_sev), ("Northumbria terrain", df_nor),
                  ("ERA5 features", df_era5)]:
    n = df.isnull().sum().sum()
    print(f"  {name}: {n} nulls {'(PASS)' if n == 0 else '(FAIL)'}")

# ── Check 2: Target variable distributions ──
print(f"\n{'='*60}")
print("  CHECK 2: Target variable class distribution")
print("="*60)
risk_cols = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]

for name, df in [("SEVERN (train)", df_sev), ("NORTHUMBRIA (test)", df_nor)]:
    print(f"\n  {name}:")
    for col in risk_cols:
        counts = df[col].value_counts().sort_index()
        total = len(df)
        dist = ", ".join([f"{k}:{100*v/total:.1f}%" for k, v in counts.items()])
        print(f"    {col}: {dist}")

# ── Check 3: Feature ranges are consistent across regions ──
print(f"\n{'='*60}")
print("  CHECK 3: Feature range comparison")
print("="*60)
feature_cols = ["dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"]

for col in feature_cols:
    s_min, s_max = df_sev[col].min(), df_sev[col].max()
    n_min, n_max = df_nor[col].min(), df_nor[col].max()
    print(f"  {col:20s}  Severn=[{s_min:.2f}, {s_max:.2f}]  "
          f"Northumbria=[{n_min:.2f}, {n_max:.2f}]")

# ── Check 4: No data leakage (ERA5 regions are separate) ──
print(f"\n{'='*60}")
print("  CHECK 4: ERA5 region separation")
print("="*60)
print(f"  ERA5 regions: {df_era5['region'].tolist()}")
print(f"  Severn ERA5 tp_mean_p99:      {df_era5.loc[df_era5['region']=='severn', 'tp_mean_p99'].values[0]:.6f}")
print(f"  Northumbria ERA5 tp_mean_p99: {df_era5.loc[df_era5['region']=='northumbria', 'tp_mean_p99'].values[0]:.6f}")
print(f"  Values differ: {df_era5.loc[df_era5['region']=='severn', 'tp_mean_p99'].values[0] != df_era5.loc[df_era5['region']=='northumbria', 'tp_mean_p99'].values[0]} (PASS if True)")

del df_sev, df_nor

  FILE SUMMARY
  severn_terrain_ready.parquet:      (35278130, 13)
  northumbria_terrain_ready.parquet:  (21391824, 13)
  era5_features.parquet:             (2, 103)

  CHECK 1: Null counts
  Severn terrain: 0 nulls (PASS)
  Northumbria terrain: 0 nulls (PASS)
  ERA5 features: 0 nulls (PASS)

  CHECK 2: Target variable class distribution

  SEVERN (train):
    risk_0_2m: 0:90.6%, 1:2.4%, 2:2.2%, 3:1.2%, 4:3.6%
    risk_0_3m: 0:90.7%, 1:3.0%, 2:1.9%, 3:1.1%, 4:3.3%
    risk_0_6m: 0:90.8%, 1:4.2%, 2:1.5%, 3:0.9%, 4:2.6%
    risk_0_9m: 0:90.9%, 1:5.1%, 2:1.2%, 3:0.7%, 4:2.1%
    risk_1_2m: 0:91.3%, 1:5.5%, 2:1.0%, 3:0.5%, 4:1.7%

  NORTHUMBRIA (test):
    risk_0_2m: 0:93.7%, 1:1.2%, 2:1.8%, 3:1.0%, 4:2.4%
    risk_0_3m: 0:93.7%, 1:1.5%, 2:1.7%, 3:0.9%, 4:2.2%
    risk_0_6m: 0:93.9%, 1:2.3%, 2:1.4%, 3:0.7%, 4:1.7%
    risk_0_9m: 0:93.9%, 1:3.0%, 2:1.1%, 3:0.6%, 4:1.4%
    risk_1_2m: 0:93.9%, 1:3.5%, 2:0.8%, 3:0.5%, 4:1.2%

  CHECK 3: Feature range comparison
  dtm_zscore            Severn=